# Varyans-Kovaryans Matrisi
Şimdiye kadar hep **tek bir değişkenin** varyansına (Konu 12) veya **iki değişken arasındaki kovaryansa** tek tek bakıyorduk. Varyans-Kovaryans Matrisi, **birden fazla değişkenin** varyans ve kovaryanslarını **tek bir tabloda** topluca gösterir — MANOVA'nın (bir sonraki konumuz) matematiksel temelini oluşturur.

## Yapısı
$n$ tane değişken varsa, $n \times n$ boyutunda bir matristir:
- **Köşegen (diagonal) elemanlar:** Her değişkenin **kendi varyansı**.
- **Köşegen dışı elemanlar:** Değişken çiftleri arasındaki **kovaryans** (iki değişkenin birlikte nasıl değiştiği)

**Örnek — 3 değişken (Satış, Kâr, İndirim) için:**
$$
\Sigma = \begin{pmatrix}
Var(Satış) & Cov(Satış,Kâr) & Cov(Satış,İndirim) \\
Cov(Kâr,Satış) & Var(Kâr) & Cov(Kâr,İndirim) \\
Cov(İndirim,Satış) & Cov(İndirim,Kâr) & Var(İndirim)
\end{pmatrix}
$$

## Simetrik Bir Matristir
$Cov(A,B) = Cov(B,A)$ olduğu için (kovaryansın sırası önemli değil), matris her zaman **köşegene göre simetriktir** — üst üçgen ile alt üçgen birbirinin aynısıdır.

## Neden MANOVA İçin Gerekli?
MANOVA (Multivariate ANOVA), **birden fazla bağımlı değişkeni aynı anda** test eder (örn: hem "memnuniyet" hem "sadakat puanı" aynı anda). Bunu yapabilmek için, değişkenlerin **birbirleriyle nasıl ilişkili olduğunu** (kovaryans yapısını) bilmesi gerekir — tek tek varyanslara bakmak yeterli değildir, çünkü değişkenler arasındaki **ilişkiyi** göz ardı eder.

## Python'da Kullanımı
```python
import numpy as np
kovaryans_matrisi = np.cov(df[['Sales','Profit','Discount']].T)
```
veya pandas ile daha okunabilir:
```python
df[['Sales','Profit','Discount']].cov()
```

## Yorumlama
- Köşegen elemanlar büyükse → o değişken kendi içinde çok yayılmış
- Köşegen dışı eleman pozitifse → iki değişken **birlikte artıyor/azalıyor**
- Köşegen dışı eleman negatifse → biri artarken diğeri **azalıyor**
- Köşegen dışı eleman sıfıra yakınsa → aralarında **doğrusal bir ilişki yok** (bağımsız olabilirler, ama kesin değil.)

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Reklam Harcaması ile Satış Adedi arasında bilerek pozitif ilişki kurguladık
reklam_harcamasi = pd.Series(np.random.normal(loc=500, scale=100, size=30), name='Reklam_Harcamasi')
satis_adedi = pd.Series(reklam_harcamasi * 0.8 + np.random.normal(0, 50, 30), name='Satis_Adedi')

# Müşteri Şikayeti, diğerlerinden bağımsız/rastgele
musteri_sikayeti = pd.Series(np.random.normal(loc=10, scale=3, size=30), name='Musteri_Sikayeti')

df = pd.concat([reklam_harcamasi, satis_adedi, musteri_sikayeti], axis=1)
kovaryans_matrisi = df.cov()
print(kovaryans_matrisi)

                  Reklam_Harcamasi  Satis_Adedi  Musteri_Sikayeti
Reklam_Harcamasi       8100.115694  6894.205135         -2.881065
Satis_Adedi            6894.205135  8014.032430          2.082044
Musteri_Sikayeti         -2.881065     2.082044          8.856273


### Sonuç
Varyans, standart sapmanın karesidir. Reklam harcamalarının standart sapması 90 (varyansı 8100), satış adedininki de yaklaşık 90 (varyansı 8014) iken, müşteri şikayetlerinin standart sapması yalnızca 3 civarındadır (varyansı 8.86). Bu durumda reklam harcaması ile satış adedinin birbirine yakın ve oldukça geniş bir yayılıma sahip olduğunu, müşteri şikayetinin ise çok daha dar bir aralıkta değiştiğini söyleyebiliriz.

Değişkenler arası kovaryansa baktığımızda, reklam harcaması ile satış adedi arasında güçlü bir pozitif ilişki (6894.21) göze çarpmaktadır — reklam harcaması artarken satış adedi de artmaktadır. Reklam harcaması ile müşteri şikayeti arasında zayıf negatif (-2.88), müşteri şikayeti ile satış adedi arasında ise zayıf pozitif (2.08) bir kovaryans bulunmuştur. Ancak bu iki değerin, Reklam-Satış kovaryansına (6894.21) kıyasla son derece küçük olması, aralarında anlamlı bir ilişki olmadığına, gözlenen küçük değerlerin rastgele varyasyondan kaynaklanabileceğine işaret etmektedir. Bu ham kovaryans değerlerinin büyüklüğünü doğru yorumlayabilmek ve ilişkilerin gerçek gücünü karşılaştırılabilir hale getirmek için değerlerin standartlaştırılması gerekir; bu işleme korelasyon denir ve ileride detaylı olarak ele alınacaktır.